In [3]:
from pathlib import Path

import pandas as pd

# Lokasi file
file_fitur = Path(
    r"D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\hasil_fitur_konvensional.csv"
)
file_target = Path(
    r"D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv\sel mikroalga\gabungan_sensor_cell_density_with_image.csv"
)
file_hasil = file_fitur.with_name("dataset_fitur_jumlah_sel_dari_sel_mikroalga.csv")

kolom_nama_gambar_fitur = "file_name"
kolom_nama_gambar_target = "Nama_Gambar"
kolom_jumlah_sel = "sensor_1_bga(cells/mL)"

# Baca sebagai teks supaya nama gambar dan angka bertitik tidak berubah format
df_fitur = pd.read_csv(file_fitur, encoding="utf-8-sig", dtype=str)
df_target = pd.read_csv(file_target, encoding="utf-8-sig", dtype=str)

# Periksa kolom yang dibutuhkan
for kolom in (kolom_nama_gambar_fitur,):
    if kolom not in df_fitur.columns:
        raise KeyError(f"Kolom '{kolom}' tidak ditemukan di file fitur.")

for kolom in (kolom_nama_gambar_target, kolom_jumlah_sel):
    if kolom not in df_target.columns:
        raise KeyError(f"Kolom '{kolom}' tidak ditemukan di file jumlah sel.")

kolom_fitur = [
    kolom for kolom in df_fitur.columns
    if kolom.startswith("fitur_")
]
if not kolom_fitur:
    raise ValueError("Tidak ditemukan kolom fitur dengan awalan 'fitur_'.")

# Buat key nama file yang dinormalisasi untuk proses pencocokan
df_fitur["_key_gambar"] = (
    df_fitur[kolom_nama_gambar_fitur].astype("string").str.strip().str.lower()
)
df_target["_key_gambar"] = (
    df_target[kolom_nama_gambar_target].astype("string").str.strip().str.lower()
)

if df_fitur["_key_gambar"].isna().any() or df_fitur["_key_gambar"].eq("").any():
    raise ValueError("Ada baris fitur yang tidak memiliki nama file gambar.")
if df_target["_key_gambar"].isna().any() or df_target["_key_gambar"].eq("").any():
    raise ValueError("Ada baris target yang tidak memiliki Nama_Gambar.")

# Pastikan setiap gambar hanya muncul satu kali pada masing-masing file
if df_fitur["_key_gambar"].duplicated().any():
    raise ValueError("Ada nama gambar duplikat di file fitur.")
if df_target["_key_gambar"].duplicated().any():
    raise ValueError("Ada Nama_Gambar duplikat di file jumlah sel.")

# Tangani format angka Indonesia (425.584,73) dan format desimal titik (425584.73).
# Koma berarti pemisah desimal: titik ribuan dibuang lalu koma diganti titik.
jumlah_sel_teks = df_target[kolom_jumlah_sel].astype("string").str.strip()
decimal_comma = jumlah_sel_teks.str.contains(",", regex=False).fillna(False)
jumlah_sel_teks.loc[decimal_comma] = (
    jumlah_sel_teks.loc[decimal_comma]
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

# Tanpa koma, pola seperti 1.234.567 dianggap memakai titik ribuan.
dot_thousands = (
    ~decimal_comma
    & jumlah_sel_teks.str.fullmatch(r"\d{1,3}(?:\.\d{3})+").fillna(False)
)
jumlah_sel_teks.loc[dot_thousands] = jumlah_sel_teks.loc[dot_thousands].str.replace(
    ".", "", regex=False
)
df_target[kolom_jumlah_sel] = pd.to_numeric(jumlah_sel_teks, errors="raise")

# Gabungkan fitur dan target berdasarkan nama gambar
df_hasil = df_fitur[
    [kolom_nama_gambar_fitur, "_key_gambar", *kolom_fitur]
].merge(
    df_target[["_key_gambar", kolom_jumlah_sel]],
    on="_key_gambar",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

# Hentikan proses jika ada gambar yang tidak berpasangan
tidak_cocok = df_hasil["_merge"] != "both"
if tidak_cocok.any():
    contoh = df_hasil.loc[tidak_cocok, "_key_gambar"].head(10).tolist()
    raise ValueError(
        f"Ada {tidak_cocok.sum()} gambar yang tidak memiliki pasangan. "
        f"Contoh nama file: {contoh}"
    )

# Simpan hanya nama gambar, fitur, dan target jumlah sel
df_hasil = df_hasil[
    [kolom_nama_gambar_fitur, *kolom_fitur, kolom_jumlah_sel]
]

df_hasil.to_csv(file_hasil, index=False, encoding="utf-8-sig")

print(f"Jumlah baris hasil: {len(df_hasil)}")
print(f"Jumlah fitur per gambar: {len(kolom_fitur)}")
print(f"File hasil disimpan di: {file_hasil}")
display(df_hasil.head())

Jumlah baris hasil: 253
Jumlah fitur per gambar: 74
File hasil disimpan di: D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\dataset_fitur_jumlah_sel_dari_sel_mikroalga.csv


,file_name,fitur_mean_R,fitur_std_R,fitur_mean_G,fitur_std_G,fitur_mean_B,fitur_std_B,fitur_mean_luminance,fitur_std_luminance,fitur_HSV_mean_H,...,fitur_GLCM_contrast_std,fitur_GLCM_dissimilarity_mean,fitur_GLCM_dissimilarity_std,fitur_GLCM_homogeneity_mean,fitur_GLCM_homogeneity_std,fitur_GLCM_energy_mean,fitur_GLCM_energy_std,fitur_GLCM_correlation_mean,fitur_GLCM_correlation_std,sensor_1_bga(cells/mL)
0,0001_K10_obsbot_photo_20260415_150013_1920x108...,44.00995,3.823578297550607,35.0767,4.404193128145041,26.25525,4.813558708222016,36.33900026,4.278307593945523,29.868131637573242,...,0.0037635383697430433,0.02823318855584455,0.0037635383697430433,0.9858834057220777,0.0018817691848714044,0.921695688139277,0.001500126944784491,0.7826353977793462,0.03032010167427576,425584.73
1,0002_K10_obsbot_photo_20260416_110013_1920x108...,50.541125,5.954801317791803,26.5739,7.149967747479705,18.22965,7.610033566121769,31.066877185000003,6.7392511272217375,15.266072273254395,...,0.014596687922074424,0.10808865306431656,0.014596687922074424,0.9459556734678417,0.007298343961037287,0.6113607243326211,0.00832017562879955,0.8433347361129755,0.02135277165012908,424919.12
2,0003_K10_obsbot_photo_20260417_115513_1920x108...,37.987725,7.432666030730495,18.6338,8.140368392155235,14.84335,8.214411773066894,22.474773965,7.736857048018734,13.503377914428711,...,0.0034199996878470076,0.022267777328855327,0.0034199996878470076,0.9888661113355721,0.0017099998439234116,0.8822082443829437,0.0011130995579341333,0.9437921121849819,0.008765994041573591,508636.97
3,0004_K10_obsbot_photo_20260418_115513_1920x108...,71.469525,8.651790061852807,26.463125,6.53682187568049,17.3195,7.20732056661836,35.371315915000004,6.150293281299252,10.069689750671387,...,0.010159305999365839,0.0787140160602005,0.010159305999365839,0.9606429919698998,0.005079652999682895,0.82203679366793,0.00449940834927176,0.7540345428801685,0.03254136153280554,508601.33
4,0005_K10_obsbot_photo_20260419_101513_1920x108...,79.338575,11.39749498659135,32.311575,5.89419596038128,22.462025,5.538488773968491,41.59837769,6.606525717981446,10.282767295837402,...,0.011047949677895942,0.0804857516224338,0.011047949677895942,0.9597571241887831,0.005523974838947984,0.6666156396185889,0.006211033334472215,0.8548385698284541,0.020113544672880047,448780.92
